In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/results.csv")

# safety: numeric + avoid division by zero
df["makespan"] = pd.to_numeric(df["makespan"], errors="coerce")
df["total_work_content"] = pd.to_numeric(df["total_work_content"], errors="coerce")
df = df.dropna(subset=["makespan", "total_work_content"])
df = df[df["makespan"] > 0]

# ThroughputIndex
df["throughput_index"] = df["total_work_content"] / df["makespan"]


In [7]:
import matplotlib.pyplot as plt

def boxplot_metric(df, experiment, metric, title, filename):
    sub = df[df["experiment"] == experiment].copy()
    versions = list(sub["experiment_version"].unique())  # oder sortiert nach Wunsch

    x = [sub[sub["experiment_version"] == v]["throughput_index"].values for v in versions]

    fig, ax = plt.subplots(figsize=(8,4))
    ax.boxplot(x, labels=versions)
    ax.set_ylabel("Throughput Index")
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)
    fig.tight_layout()
    fig.savefig(filename)
    plt.close(fig)

boxplot_metric(
    df,
    experiment="Experiment 1",
    metric="throughput_index",
    title="Throughput Index by Line Configuration",
    filename="diagrams/throughput_index_exp1.png"
)

C:\Users\nicol\AppData\Local\Temp\ipykernel_21316\4119709851.py:10: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(x, labels=versions)


In [9]:
import ast

def parse_usage_col(df):
    df = df.copy()
    df["usage"] = df["usage"].apply(lambda s: ast.literal_eval(s) if isinstance(s, str) else s)
    # average utilization excluding first/last if you want
    def avg_util(run_usage):
        rels = [u[2] for u in run_usage]
        if len(rels) > 2:
            rels = rels[1:-1]
        return np.mean(rels) if rels else np.nan
    df["avg_utilization"] = df["usage"].apply(avg_util)
    return df

df2 = parse_usage_col(df)

sub = df2[df2["experiment"] == "Experiment 3"].copy()

fig, ax = plt.subplots(figsize=(6,4))
for v in sub["experiment_version"].unique():
    s = sub[sub["experiment_version"] == v]
    ax.scatter(s["avg_utilization"], s["throughput_index"], label=v, alpha=0.6)

ax.set_xlabel("Average utilization")
ax.set_ylabel("Throughput Index")
ax.set_title("Utilization vs Throughput Index")
ax.legend()
fig.tight_layout()
fig.savefig("diagrams/util_vs_throughput_exp3.png")
plt.close(fig)

In [ ]:
# gap
import pandas as pd
import numpy as np
results1 = pd.read_csv("data/results.csv")
results2 = pd.read_csv("data/results2.csv")

print(np.mean(results1["gap"]))
print(np.mean(results2["gap"]))
print(np.mean(results2.loc[results2["experiment_version"].str.contains("Size = 15"), "gap"]))
print(np.mean(results2.loc[results2["experiment_version"].str.contains("Size = 30"), "gap"]))
print(np.mean(results2.loc[results2["experiment_version"].str.contains("Size = 45"), "gap"]))
print(np.mean(results2.loc[results2["experiment_version"].str.contains("3 Layers"), "gap"]))
print(np.mean(results2.loc[results2["experiment_version"].str.contains("4 Layers"), "gap"]))
print(np.mean(results2.loc[results2["experiment_version"].str.contains("5 Layers"), "gap"]))
print(np.mean(results2.loc[results2["experiment_version"].str.contains("6 Layers"), "gap"]))

0.0
0.012615952981208723
0.0
0.0003472222222222208
0.03760373529921056
0.04165846203703046
0.000193723363037582
0.0
0.008796296296296295


In [7]:
# runtime

print(np.mean(results1.loc[results1["experiment"] == "Experiment 1", "runtime"]))
print(np.mean(results1.loc[results1["experiment"] == "Experiment 2", "runtime"]))
print(np.mean(results1.loc[results1["experiment"] == "Experiment 3", "runtime"]))

print(np.mean(results2["runtime"]))

4.4909888982772825
12.253722225295173
3.4048666755358377
161.4886638879776


In [ ]:
import pandas as pd
import numpy as np

results1 = pd.read_csv("data/results.csv")
results2 = pd.read_csv("data/results2.csv")  # Experiment 4 (Batch extension)

def solver_stats(df, group_cols):
    d = df.copy()
    d["runtime"] = pd.to_numeric(d["runtime"], errors="coerce")
    d["gap"] = pd.to_numeric(d["gap"], errors="coerce").fillna(0.0)

    def q25(x): return np.nanquantile(x, 0.25)
    def q50(x): return np.nanquantile(x, 0.50)
    def q75(x): return np.nanquantile(x, 0.75)

    out = (d.groupby(group_cols)
             .agg(
                 n=("runtime", "size"),
                 runtime_median=("runtime", q50),
                 runtime_q25=("runtime", q25),
                 runtime_q75=("runtime", q75),
                 runtime_max=("runtime", "max"),
                 gap_mean=("gap", "mean"),
                 gap_median=("gap", q50),
                 share_gap_pos=("gap", lambda s: float(np.mean(s > 1e-12))),  # % runs with gap>0
                 share_gap_gt1pct=("gap", lambda s: float(np.mean(s > 0.01))), # % runs with gap>1%
             )
             .reset_index())
    out["runtime_iqr"] = out["runtime_q75"] - out["runtime_q25"]
    return out.sort_values(group_cols)

# results1: Experiments 1–3 by configuration
stats_1to3 = solver_stats(results1, ["experiment", "experiment_version"])
print(stats_1to3)

# results2: Experiment 4 by line-layer and batch size 
tmp = results2.copy()
tmp[["line", "batch"]] = tmp["experiment_version"].str.split("-", n=1, expand=True)
tmp["line"] = tmp["line"].str.strip()
tmp["batch"] = tmp["batch"].str.strip()

stats_exp4 = solver_stats(tmp, ["line", "batch"])
print(stats_exp4)

     experiment experiment_version   n  runtime_median  runtime_q25  \
0  Experiment 1   High Flexibility  30          9.1670      7.70700   
1  Experiment 1    Low Flexibility  30          1.9435      1.34200   
2  Experiment 1     No Flexibility  30          0.2025      0.16500   
3  Experiment 2           4 Layers  30          3.8605      3.19650   
4  Experiment 2           5 Layers  30          9.8025      8.71200   
5  Experiment 2           6 Layers  30         13.7165     11.01400   
6  Experiment 3           3 Layers  30          3.8820      3.28725   
7  Experiment 3           4 Layers  30          2.0445      1.30625   
8  Experiment 3           5 Layers  30          3.0095      2.24700   
9  Experiment 3           6 Layers  30          4.2260      2.95475   

   runtime_q75  runtime_max  gap_mean  gap_median  share_gap_pos  \
0     11.39600       39.349       0.0         0.0            0.0   
1      2.93325        4.944       0.0         0.0            0.0   
2      0.26650